In [10]:
import numpy as np 
import matplotlib.pyplot as plt
import OrcFxAPI
import h5py
import pandas as pd
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from scipy.optimize import minimize


In [11]:
model_path = r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Harlequin_fixed_40s.dat"
sim_path = r"c:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\Sim_data\Masscheck_V1"

lookback_window = 5.0      # seconden vóór decay-start waarin we de amplitude zoeken
quiet_window_end = 12.0     # rustige periode eindigt 12 s vóór decay-start
quiet_window_length = 50.0


In [12]:
model = OrcFxAPI.Model(model_path)
constraint = model["decay_constraint"]
floaters = model["floaters"]
floatertype = model["Floatertype"]

In [13]:
constraint.InFrameInitialZ = 2
constraint.InFrameInitialX = 0
constraint.InFrameInitialY = 0
constraint.InFrameInitialAzimuth = 0
constraint.InFrameInitialDeclination = 0 
constraint.InFrameInitialGamma = 0

floatertype.OtherDampingLinearCoeffx = 0 #surge
floatertype.OtherDampingLinearCoeffy = 0 #sway
floatertype.OtherDampingLinearCoeffz  = 0 #heav
floatertype.OtherDampingLinearCoeffRx = 0 #roll
floatertype.OtherDampingLinearCoeffRy = 0 #pitch
floatertype.OtherDampingLinearCoeffRz = 0 #yaw

floatertype.OtherDampingQuadraticCoeffx = 0 #surge
floatertype.OtherDampingQuadraticCoeffy = 0 #sway 
floatertype.OtherDampingQuadraticCoeffz = 0 #heav
floatertype.OtherDampingQuadraticCoeffRx = 0#roll
floatertype.OtherDampingQuadraticCoeffRy = 0 #pitch
floatertype.OtherDampingQuadraticCoeffRz = 0 #yaw


In [14]:
model.RunSimulation()
print("Simulatie klaar.")

Simulatie klaar.


In [15]:
def compute_period(t, z):
    peaks, _ = find_peaks(z)
    
    if len(peaks) < 2:
        return None  # geen betrouwbare periode
    
    T = np.diff(t[peaks])
    return np.mean(T)

In [16]:
target_T = 4.02

mass_values = np.linspace(72,75, 5)  # kies range die logisch is
results = []

for i, mass in enumerate(mass_values):

    model = OrcFxAPI.Model(model_path)
    
    constraint = model["decay_constraint"]
    floaters = model["floaters"]
    floatertype = model["Floatertype"]

    constraint.InFrameInitialZ = 1

    floatertype.Mass = mass

    floatertype.OtherDampingLinearCoeffz = 0
    floatertype.OtherDampingQuadraticCoeffz = 0

    model.RunSimulation()

    model.SaveSimulation(f"{sim_path}\{i}_V1.dat")

    print(f"Sim {i} klaar (mass={mass})")

    

<string>:23: SyntaxWarning: invalid escape sequence '\{'
<>:23: SyntaxWarning: invalid escape sequence '\{'
<string>:23: SyntaxWarning: invalid escape sequence '\{'
<>:23: SyntaxWarning: invalid escape sequence '\{'
C:\Users\verav\AppData\Local\Temp\ipykernel_33772\663030276.py:23: SyntaxWarning: invalid escape sequence '\{'
  model.SaveSimulation(f"{sim_path}\{i}_V1.dat")


Sim 0 klaar (mass=72.0)
Sim 1 klaar (mass=72.75)
Sim 2 klaar (mass=73.5)
Sim 3 klaar (mass=74.25)
Sim 4 klaar (mass=75.0)


In [17]:
results = []

for i, mass in enumerate(mass_values):

    model = OrcFxAPI.Model(f"{sim_path}\{i}_V1.dat")
    floaters = model["floaters"]

    t_sim = model.general.TimeHistory("Time")
    z_sim = floaters.TimeHistory("Z")

    T = compute_period(t_sim, z_sim)

    results.append((mass, T));

<string>:5: SyntaxWarning: invalid escape sequence '\{'
<>:5: SyntaxWarning: invalid escape sequence '\{'
<string>:5: SyntaxWarning: invalid escape sequence '\{'
<>:5: SyntaxWarning: invalid escape sequence '\{'
C:\Users\verav\AppData\Local\Temp\ipykernel_33772\1674216355.py:5: SyntaxWarning: invalid escape sequence '\{'
  model = OrcFxAPI.Model(f"{sim_path}\{i}_V1.dat")


In [18]:
best = min(results, key=lambda x: abs(x[1] - target_T))

print("Beste resultaat:")
print(f"massa = {best[0]:.6f}")
print(f"periode = {best[1]:.6f}")
print(f"afwijking = {abs(best[1] - target_T):.6f}")


Beste resultaat:
massa = 72.000000
periode = 4.070000
afwijking = 0.050000
